# 🗃️ **Backwards Alibi**


## What Garbage Do We Have Here

Alright, settle in. Today's patient is a Titanic survival classifier, the "hello world" of ML, the training wheels, the thing so simple you could practically eyeball the answer. Class + Sex + a bit of Age and Fare, predict who lived. A bright first-year could hand-code a decent version of this on a napkin.

And yet. When you run this notebook, your logistic regression is going to shuffle in, look at a test set it's never seen, and produce accuracy that's barely better than *guessing "everyone died"* ... which, charming as that strategy is, is not what we pay machine learning models to do. Somewhere between "clean data" and "trained model," this thing lost the plot entirely.

The suspicious part isn't that it's wrong. Garbage models are wrong all the time, that's basically the whole genre. The suspicious part is *how consistently, boringly wrong* it is like it's not confused so much as confidently answering a slightly different question than the one you asked it. That's not noise. That's a pattern. And patterns mean SOMEONE'S GETTING FIRED (eh if you know the ref you know)


Test accuracy lands around ~0.63-0.66  which is only marginally above the "always predict died" baseline of ~0.62. A logistic regression on
Pclass/Sex/Age/Fare should comfortably clear ~0.78-0.80 on this dataset. If your accuracy is suspiciously close to the majority-class
baseline... ummm yikes


## Run It

Just run every cell top to bottom. Don't skip ahead, don't get clever. Watch the final accuracy print out and try not to wince.


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

DATA_PATH = "./data/Titanic-Dataset.csv"
RANDOM_STATE = 42


Loading the CSV. Groundbreaking stuff. Try to contain your excitement for the next four cells, we're about to do some genuinely boring, genuinely necessary cleaning. Boring is good. Boring means nobody's lying to you yet.


In [2]:
def load_data(path):
    """Load the raw Titanic CSV into a DataFrame."""
    df = pd.read_csv(path)
    return df


df_raw = load_data(DATA_PATH)
df_raw.head()


FileNotFoundError: [Errno 2] No such file or directory: './data/Titanic-Dataset.csv'

Now we clean. Median-impute Age because a chunk of passengers apparently didn't feel like sharing their birthday with the ship's clerk, fill the two missing Embarked values with whatever port shows up most, and encode Sex and Embarked as numbers because scikit-learn does not care about your feelings, only your floats.


In [ ]:
def prepare_features(df):
    """Clean and encode the feature matrix used for training."""
    df = df.copy()
    df["Age"] = df["Age"].fillna(df["Age"].median())
    df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])
    df["Sex"] = df["Sex"].map({"male": 0, "female": 1})
    df["Embarked"] = df["Embarked"].map({"S": 0, "C": 1, "Q": 2})

    feature_cols = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]
    return df[feature_cols].reset_index(drop=True)


X = prepare_features(df_raw)
X.head()


Alright, features are clean, indices are reset, everybody's `.reset_index(drop=True)`'d like responsible adults. Now — labels. This part's basically a formality. It's a column. We're grabbing a column.

...I added a little defensive helper here too, because I got burned once by an export job that leaked info across adjacent rows in a raw dump, and now I guard against it everywhere out of pure trauma. You're welcome. Very safety-conscious of me, really.


In [ ]:
def make_labels(df):
    """Build the target vector aligned with the cleaned feature set."""
    y = df["Survived"]
    # guard against accidental leakage from adjacent rows in the raw export
    
    return y.reset_index(drop=True)


y = make_labels(df_raw)
y.head()


Split, train, evaluate. Standard-issue logistic regression, nothing fancy, no regularization drama, no hyperparameter sweep. If the pipeline upstream is honest, this thing should read the data like a decent detective reads a room.

...it is not going to read the room.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

preds = clf.predict(X_test)

print("Baseline (always predict majority class):", round(1 - y.mean(), 4))
print("Test accuracy:", round(accuracy_score(y_test, preds), 4))
print("Confusion matrix:\n", confusion_matrix(y_test, preds))


## The Math Clue

Okay. Coffee down. Let's actually talk, because this one's worth understanding properly and I'm not going to hand-wave it at you.

**Supervised learning is a claim about a *joint* distribution.** When you fit a model on pairs $(x_i, y_i)$, you are implicitly assuming those pairs are draws from some joint distribution $P(X, Y)$ and critically, that the *pairing itself* is correct. Row $i$'s features actually belong to row $i$'s label. The whole enterprise of "learning a relationship between X and Y" only means something if that correspondence holds.

Now ask: what happens if $X$ and $Y$ are statistically **independent** i.e. $P(X, Y) = P(X)\,P(Y)$, knowing $X$ tells you *nothing* about $Y$? In that world, the Bayes-optimal classifier (the best any model can do, even with infinite data and a perfect algorithm) degenerates to something almost insulting in its simplicity:

$$
\hat{y}^{*} = \arg\max_{y} P(Y = y)
$$

In words: if features carry zero information about the label, your best strategy is to *ignore the features entirely* and just always predict whichever class is more common overall. The achievable accuracy caps out at:

$$
\text{Accuracy}_{\max} = \max_{y}\, P(Y = y)
$$

For this dataset, $P(\text{Survived}=0) \approx 0.616$ "everybody died" is right about 62% of the time just by prior probability, no features required. A model that's only *barely* beating that number isn't a weak model. It is...

Ha! That's for you to find! Maybe while you're at it look at permutation tests. The vibes are similar



## Your Mission

Somewhere in the four functions above, one passenger's features are wearing another passenger's alibi. Go find whose. The math above tells you *what* broke the correspondence — it does not tell you *where*. That part's on you. Read every line like you don't trust it, because at least one of them has earned that.


## Submission

Fork this repo → fix the bug → open a PR → wait for a human to confirm the fix is correct and the symptom is resolved → receive your title.

### On confirmed fix: 🏅 Sergeant of the Off-By-One Squad, Bureau of Suspiciously Round Numbers

Zane, standing on a chair for no reason, saluting incorrectly: You found it. By the powers vested in me by absolutely nobody, I hereby promote you to Sergeant of the Off-By-One Squad. Your jurisdiction covers shifted indices, misaligned joins, and anyone who writes 'defensive' in a comment when they mean 'catastrophic.' Badge is metaphorical. Pride is real. Go be insufferable about it.